# Connectomics Exercise 1: Which interneurons target which compartments?

A pyramidal cell is not inhibited uniformly. Different inhibitory subclasses place their
synapses in different places on it, and that arrangement is a large part of what those
subclasses *are*. We build that  **L5-ET** cells — thick-tufted, subcortically
projecting, with some of the most elaborate dendrites in cortex.

We will build also everything directly from CAVE to get used to working with direct data access.

Disclaimer: Claude was used to help generate exercise details and solutions.

In [ ]:
import sys
from os.path import join as pjoin

mat_version = 1196
sys.path.append(pjoin("..", "utils"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import ossify as osy
from caveclient import CAVEclient
from standard_transform.datasets import v1dd_ds

from utils import cell_type_palette

sns.set_context("notebook")
plt.rcParams["figure.dpi"] = 110

If you have not set up a CAVE token yet, uncomment the `setup_token` line, run it once,
and follow the instructions.


In [ ]:
# CAVEclient.setup_token("https://global.em.brain.allentech.org")
client = CAVEclient(datastack_name="v1dd_public", version=mat_version)

### The CAVE calls you will need

Every query in this notebook is one of four shapes. The point of the exercises is the analysis,
not remembering the API, so here they all are.

**Discover what exists.**

```python
client.materialize.get_tables()          # queryable tables
client.materialize.get_views()           # views (pre-joined convenience tables)
client.materialize.get_versions()        # materialization versions
```

**Pull a whole table.** The `tables.<name>()` accessor takes filters as keyword arguments;
`.get_all()` skips the filter and returns everything.

```python
df = client.materialize.tables.cell_type_multifeature_v1().get_all(
    split_positions=True,          # pt_position -> pt_position_x / _y / _z
    desired_resolution=[1, 1, 1],  # positions in NANOMETRES rather than voxels
)

df = client.materialize.tables.proofreading_status_and_strategy(
    status_axon=True,              # any column can be a filter kwarg
).query(split_positions=True)
```

**Query synapses.** `pre_ids` and `post_ids` each take a list of root ids. Giving both is a
logical AND — only synapses from those cells onto those cells — which is how you ask a narrow
question cheaply instead of filtering a huge answer afterwards.

```python
syn = client.materialize.synapse_query(
    post_ids=some_root_ids,                  # or pre_ids=..., or both
    desired_resolution=[1, 1, 1],
    split_positions=True,
)
```

**Query a table by an explicit list of ids.** Needed for reference tables such as the synapse
compartment predictions, which are keyed to synapse ids rather than to cells.

```python
client.materialize.query_table(
    "synapse_target_predictions_ssa",
    filter_in_dict={"target_id": [int(i) for i in some_synapse_ids]},
    select_columns=["target_id", "tag"],
    merge_reference=False,   # return the reference table's own rows, unjoined
    log_warning=False,
)
```

Two things that will bite you regardless of which shape you use:

- **Root ids come back as a nullable `Int64`**, which is not JSON-serialisable. Passing
  `df.pt_root_id.values` into a query raises
  `TypeError: Object of type IntegerArray is not JSON serializable`. Use `root_ids()` below.
- **A query that is too large returns `502 Bad Gateway`**, not a helpful error. The limit is on
  the size of the response, not the number of ids. Question 4 makes you meet this properly.


### Pre-screened cells

These were chosen ahead of time — soma away from the volume edge, an apical reaching L1, all
four inhibitory subclasses present among its proofread inputs, and no merge balloon in the
radius profile. Being handed a curated example is normal practice and rarely admitted to; the
last exercise asks you to pick your own and see what goes wrong.

**These ids belong to materialization 1196.** If the dataset is rematerialized they must be
regenerated.


In [ ]:
EXEMPLAR_L5ET = 864691132734343191
BACKUP_L5ET = [864691132830244868, 864691132669405014]

The helper the note above refers to:


In [ ]:
def root_ids(series):
    """Root ids as a plain uint64 array, safe to hand to a CAVE query."""
    return series.dropna().astype("uint64").to_numpy()

**Coordinates come in three flavours and mixing them fails silently.** Voxels, raw
nanometres, and *transformed* microns (rotated so the pia is flat, with y as cortical depth).
The last is the most meaningful for geometry — and subtracting a raw-µm position from a
transformed-µm one adds a large fixed offset with no error and no warning. Push every
transform through one helper so it cannot happen:

In [ ]:
def to_transformed_um(df, prefix):
    """Columns f'{prefix}_x/y/z' in nanometres -> transformed microns, pia-flat."""
    cols = [f"{prefix}_{ax}" for ax in "xyz"]
    return v1dd_ds.transform("nm").apply(df[cols].values)

**A synapse's position relative to the cell it lands on.** Question 3 will ask you to compute the location of a synapse relative to the cell body in spherical coordinates. Feel free to write your own function, but rather than get hung up on geometry you can also use this function to accomplish that. Given a set of points and a soma, both in transformed µm, it returns how far each point is from that soma and in what
direction — with the compass flattened to a single angle.


In [ ]:
def r_theta(points_um, soma_um):
    """Spherical-ish coordinates about a soma, with azimuth dropped.

    r     : straight-line distance from the soma, µm
    theta : degrees out of the tangential plane; +90 = straight toward the pia,
            0 = level with the soma, -90 = straight down.

    Dropping azimuth assumes the target is roughly rotationally symmetric about the
    pia axis -- fair for a pyramidal cell, wrong for anything with a planar arbor.
    """
    delta = points_um - soma_um
    lateral = np.hypot(delta[:, 0], delta[:, 2])
    return (
        np.linalg.norm(delta, axis=1),
        np.degrees(np.arctan2(-delta[:, 1], lateral)),
    )

**Plotting that frame.** `cloud` draws one panel of synapses in a target's own frame. It is
here so the figures later stay short; read it when you first use it in Question 4.


In [ ]:
def cloud(ax, df, title, color):
    """Every synapse in the frame of the cell it landed on. Azimuth was collapsed, so
    lateral distance is a magnitude -- mirror it, which is what the rotational-symmetry
    assumption already claims."""
    lat = df.r_um * np.cos(np.radians(df.theta_deg))
    vert = df.r_um * np.sin(np.radians(df.theta_deg))
    ax.scatter(
        np.r_[lat, -lat], np.r_[vert, vert], s=3.0, alpha=0.18, lw=0, color=color
    )
    ax.scatter([0], [0], marker="*", s=180, c="k", zorder=6)
    for rad in (25, 50, 100, 150):
        ax.add_artist(plt.Circle((0, 0), rad, fill=False, color="0.6", lw=0.7, ls=":"))
    ax.set(
        xlim=(-160, 160),
        ylim=(-160, 200),
        aspect="equal",
        title=title,
        xlabel="lateral distance (µm)",
    )
    # The frame is clipped so the tight cloud is readable; say how much falls outside
    # rather than letting the crop quietly imply the cloud ends there.
    out = 100 * (df.r_um > 160).mean()
    if out > 1:
        ax.text(
            0.97,
            0.03,
            f"{out:.0f}% beyond frame",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
            color="0.4",
        )

---
## Question 1 — Who are the L5-ET cells, and whose reconstruction do you have to trust?

**ET** is *extratelencephalic*: They are a kind of pyramidal cell that projects out of cortex, to thalamus,
superior colliculus and brainstem. They are also known as thick-tufted cells, with a long apical trunk
rising to a (sometimes very wide) tuft in Layer 1, plus a basal skirt around the soma.

Inhibitory cells come pre-sorted into four subclasses — `PTC`, `DTC`, `ITC`, `STC` — by
`cell_type_multifeature_v1`, a classifier built on **soma, nucleus, dendrite and spine
morphology**. Note what is *not* in that list: the classifier never sees where a cell's
synapses land.

**Treat the four labels as opaque for now.** They are four groups that look different down a
microscope, nothing more. Whether they also *behave* differently — whether they put their
synapses in different places on a target — is the question this module aims to answer. The
names do stand for something, and we come back to them at the end; unpacking them now would
hand you the answer before you have measured anything.


**Task:** Build a table of all classified cells from `cell_type_multifeature_v1` and remove any duplicate root ids that do not have a well-defined type. How many of each type, and which are excitatory vs inhibitory?

Note: The table query function is `client.materialize.query_table`, or you can also use `client.materialize.tables.<table>.query`

*Solution note: Later cells expect this table to be called `cell_df`.*

In [ ]:
# Your code here

Note the spelling: **`L5ET`, no hyphen.** Other modules in this workshop load derived data
products that spell the same type `L5-ET`.

The shared palette in `utils` is keyed the hyphenated way too, so bridge the two spellings
once rather than maintaining a second dict.

In [ ]:
import re  # Regex is a tool to search for and replace strings based on matching various patterns.


def hyphenated(t):
    """CAVE spells excitatory types 'L5ET'; cell_type_palette keys them 'L5-ET'."""
    return re.sub(r"^(L\d)([A-Z]+)$", r"\1-\2", str(t))


PALETTE = {
    t: cell_type_palette.get(hyphenated(t), "0.6")
    for t in cell_df.cell_type.dropna().unique()
}
print({k: PALETTE[k] for k in list(PALETTE)[:4]})

Soma depth by cell type is worth plotting before going further — it is the sanity check that the coordinate transform is doing what it claims.

In [ ]:
# Your code here

The excitatory types stack in layer order — that is the sanity check that the coordinate
transform is doing what it claims. The inhibitory types do not, because they are defined by
*who they target*, not by where they sit.


**Task:** Load `EXEMPLAR_L5ET` and plot it. Argue from the picture that it is thick-tufted.


In [ ]:
# Your code here

The apical trunk runs from the soma at roughly 470 µm up to a tuft that reaches the pia,
and there is a basal skirt at the soma. That is the thick-tufted ET morphology.

**The axon is in there too**, sprawling laterally. It belongs to the *presynaptic*
side of this cell's story, not the postsynaptic side we care about, and Question 3 has to
remove it before measuring anything along the dendrite.


### Whose reconstruction has to be right?

Now the part that governs every number in this module.

**Task:** Build the presynaptic universe. Join `proofreading_status_and_strategy` to the cell
type table and count, per inhibitory subclass, how many cells have `status_axon == True`.
Then look at `strategy_axon` and ask what the partially-extended cells are.

*Call the joined, filtered frame `inh_pf`.*

In [ ]:
# Your code here

**A few concepts you might have started to think about**

You are about to study synapses *onto* an L5-ET cell.

1. **Whose reconstruction quality matters, the ET cell's or the interneurons'?**
   Both, differently. The ET cell needs a trustworthy **dendrite**, which is what makes "we
   found every synapse onto it" true. The interneurons need trustworthy **axons**, which is
   what makes "this synapse came from a PTC" true.

2. **A trusted dendrite means every synapse was found. Does it mean every synapse was
   correctly attributed?**
   No. Segmentation makes **merge errors** — a fragment of another cell's axon glued onto a
   soma. The synapse is real and its location is right; the presynaptic identity is wrong.
   Proofreading removes those merges, which is why `status_axon == True` is the filter used
   throughout. Although note that small numbers of errors will remain after proofreading, so
   always double check that any result isn't based on some small number of potential artifacts.

4. **Reconstruction loses thin distal branches more often than thick proximal ones. Suppose
   one subclass tends to synapse close to its targets' somata and another travels far out
   along the dendrite. Without the filter, which of the two looks over-represented?**
   The proximal one. Truncation strips the far-travelling subclass's synapses while leaving
   the close one's largely intact, so an unfiltered analysis reports "inhibition is proximal"
   **whatever the biology does.** That is the specific reason this module filters — and note
   you cannot yet say which label is which, which is the point. Question 2 measures the size
   of the effect; Question 3 identifies the players.

5. **What does the restriction cost?**
   About 95% of the typed inhibitory cells (Figure 2). A much smaller sample, in exchange for
   labels that mean what they say.

`strategy_axon` further splits proofread cells into fully and partially extended, but that is
**completeness, not correctness** — a partially extended axon has had its merges cleaned, so
its synapses are still correctly attributed. Completeness only matters when you need a
per-axon denominator, which no question here does.

---
## Question 2 — What lands where?

Synapses are labelled by what they land on: a dendritic **spine**, a dendritic **shaft**, or
the **soma**. The distinction is electrotonic. A spine is a small, electrically isolated
compartment, so excitatory input there can summate locally. A synapse on the soma sits right
where the axon initial segment decides whether to fire, so it can shunt the whole cell.
The textbook claims, stated so we can check them:

1. Excitatory → excitatory synapses are onto **spines**.
2. Inhibitory → excitatory synapses are onto **shafts**.
3. Synapses onto inhibitory cells are onto **shafts**.
4. Basket cells target **somata** — so one of the four inhibitory labels should own most
   somatic synapses. Which one?


**Task — assemble the synapse table.** Three joins get you from "a list of L5-ET cells" to
"one row per synapse, with a compartment and a presynaptic subclass": pull the input
synapses, attach the compartment label, attach the presynaptic cell type and restrict to
proofread axons. Report the synapse count before and after that last step.

*The next three cells do them in order — read each one before running it, because the third
is a decision and the first two are only plumbing.*

*Names later cells rely on: `COHORT` and `syn` for the 15-cell query, `COHORT_WIDE`
and `syn_inh` for the two-sided one.*

In [ ]:
# Your code here

The compartment label lives in `synapse_target_predictions_ssa`, a reference table keyed on
synapse id. Note it is `_ssa`, not `_ssa_v2` — only the former exists at materialization 1196.

`fetch_tags` splits the request into chunks, which is given here rather than explained —
there is a reason the server will not take it all at once, and Question 3 makes you find it.


In [ ]:
def fetch_tags(syn_ids, chunk=40_000):
    """Compartment label (spine / shaft / soma) for each synapse id."""
    out = []
    for i in range(0, len(syn_ids), chunk):
        out.append(
            client.materialize.query_table(
                "synapse_target_predictions_ssa",
                filter_in_dict={"target_id": list(map(int, syn_ids[i : i + chunk]))},
                select_columns=["target_id", "tag"],
                merge_reference=False,
                log_warning=False,
            )
        )
    return pd.concat(out, ignore_index=True).rename(columns={"target_id": "id"})

In [ ]:
# Your code here

Now the presynaptic side, and **the one real choice in this sequence**: restricting to cells
with a proofread axon. This restricts the analysis set quite a lot, but still leaves a lot of measured inputs.

In [ ]:
# Your code here

**Task:** Build the contingency table of presynaptic type × target compartment, and
normalize it **by row**. This is a table where each row will be a cell type, each column will be a compartment, and the value will be the number of synapses in that compartment. You can then ask why the column version — "who owns this compartment?" — is
not a question you can answer here.

In [ ]:
# Your code here

**Read this table across the rows!**

A row asks **where a given type puts its synapses.** It is conditional on the type, so it does
not care how well any *other* type happens to be sampled — which is why the excitatory row and
the inhibitory rows can come from different queries and still both be sound. Each row is its
own denominator.

What aspects of this hold with the textbook and which do not?

Before doing more statistics, let's make one picture. Here are the exemplar's input synapses on its own
morphology, coloured by presynaptic subclass — one cell, not the result.

In [ ]:
# Your code here

---
## Question 3 — Where along the dendrite are these synapses?

"Shaft" covers a synapse 20 µm from the soma and one 400 µm out on the apical tuft.
Electrically those are different worlds, so a compartment label is not enough — we need a
position along the arbor.

The right measure is **geodesic distance**: path length through the dendrite from the soma. It
is what sets how much of a synapse's effect survives the trip, and it needs a **skeleton**,
which is why this question works on the one screened cell where we have one.

**Task 1 — get the distances.** Compute geodesic distance to the soma for every input
synapse on `EXEMPLAR_L5ET` — **masking to the dendrite first**, because on this cell the axon
is about a third of the skeleton and leaving it in lets a synapse be "reached" the wrong way.

*Names later cells rely on: `dend` for the masked cell, `anno` for its `post_syn` annotation,
and `syn_df` for the per-synapse frame.*

In [ ]:
# Your code here

**Task 2 — the distribution.** Histogram geodesic distance for the proofread inhibitory
inputs, split by the four subclasses. Do they sit in different places along the dendrite?

In [ ]:
# Your code here

**Task 3 — put numbers on it.** Median and quartiles of geodesic distance per subclass.

In [ ]:
# Your code here

Two cautions: **`ITC` has a single-digit count on this cell**, so
its median is a curiosity rather than a measurement; the next question fixes that by pooling
over many targets. And **a raw histogram is partly a picture of where the dendrite is** — more
synapses at 100 µm may only mean more dendrite at 100 µm. That confound does not disturb the
*ranking* here, because all subclasses are measured against the same dendrite, but it
matters the moment you want an absolute density. The optional section at the end takes this question on.

---
## Question 4 — How does this generalize across pre- and posts-synaptic cells?

Question 3 did it the right way, and the cost is visible: one skeleton, one cell. The next
question needs *hundreds* of target cells, so geodesic distance is out of reach.

The way round is a frame anchored on the target's **soma position**, which is just a column —
`r_theta` from the setup. But it is an approximation, so before spending it we measure what it
costs, on the one cell where both measures exist.

**Task 1 — how good is straight-line distance?** Compute **r** for the same synapses and
scatter it against the geodesic distance you already have.

**Predict first, in writing:** as synapses get further from the soma, does the approximation
get better or worse?

In [ ]:
# soma_um = v1dd_ds.transform("nm").apply(root_xyz.reshape(1, 3))[0]
# syn_um  = v1dd_ds.transform("nm").apply(anno.vertices)
# syn_df["r_um"], syn_df["theta_deg"] = r_theta(syn_um, soma_um)
# ...then colour the skeleton by theta and look at it.

# Your code here

**Task 2 — does the ranking survive?** Recompute the subclass comparison with **r** and
**θ** alongside geodesic. An approximation that blurs the axis but preserves the ordering is
fine for comparing populations; it would not be fine for a quantitative claim about decay.

In [ ]:
# Your code here

**Choosing the exemplars.** Filter on `status_axon` for correctness as always, then
*screen* on `axon_fully_extended` plus a high synapse count. Completeness is not needed for
the synapses to be valid — it is a quality heuristic, because an axon that stopped early
samples fewer of its targets and possibly a skewed subset.


In [ ]:
# Your code here

**Task 3 — start with one of each.** Pull the output synapses of one `PTC` and one `DTC`, and
place every synapse in the frame of *the target it landed on*. Plot the two side by side.

One cloud on its own is uninterpretable — a shape only means something against a comparison.
Two cells cost two queries and about a second, and they give you the answer before any of the
plumbing that follows.

In [ ]:
# Your code here

**Task 4 — now scale it the obvious way, and watch it break.** You need this for ~20
interneurons. Put all twenty ids into a single query. **Write down what you expect before
you run it.**

In [ ]:
# Your code here

**This will almost certainly fail!**

**It is not a bug in your code and not a bad root id.** The traceback ends in
`requests.HTTPError`, not a `KeyError` — your request was well-formed and the *server* refused
it. A 502 with HTML in the body is a proxy giving up on a response that got too large.
Recognising which layer failed is the transferable skill.

**The limit is on response size, not id count.** Twenty of these cells return fine; forty do
not. Twenty cells with bigger axons might also not. So "find a batch size that works" is a
fragile fix — pick one comfortably inside the limit and handle failure gracefully.

**Task 5 — fix it.** Batch the request and concatenate. Ten ids per call sits inside the
limit and costs a tenth of the round trips; since `synapse_query` returns `pre_pt_root_id` on
every row, a `groupby` recovers per-cell results afterwards. The design choice worth making
deliberately is what to do when a batch *does* fail — discarding all ten is wasteful when
usually nine are fine.


In [ ]:
# def collect(pre_ids, chunk=10, pause=3.0):
#     """Query in batches; on a failed batch, retry it one cell at a time.
#     Returns (frame, ids that never came back)."""
#     ... for i in range(0, len(pre_ids), chunk): ...
#     return pd.concat(frames, ignore_index=True), failed

# Your code here

Two refinements in there worth insisting on, because they are what separates a script that
survives a real dataset from one that does not. **Retry with a pause**, because a 502 can also
be transient and a twenty-minute loop that dies on iteration 17 has wasted twenty minutes. And
**collect failures rather than raising**, so one bad cell costs you one cell rather than the
whole run.

**Task 6 — do the two example cells hold up?** Redraw the same clouds from the full panel.


In [ ]:
# Your code here

**Task 7 — the repeatability question.** Pooling over targets could still hide the thing we
actually want to know. Split by target: for each (interneuron, target) pair compute a median r,
and look at the spread *within* one interneuron. Does a single cell do the same thing to all of
its targets, or is it proximal on some and distal on others?

Make a dataframe where each row is a proofread interneuron (with known type) and there are columns for the median radial distance to soma as well as the interquartile interval (25-to-75th percentile range) of the radial distance distribution.
Plot 1) a histogram of the median distance distribution by interneuron subclass and 2) a plot showing median and IQR values for each interneuron as a dot with errorbars.

In [ ]:
# Your code here

**Task 8 — split by target cell type.** Does an interneuron treat every kind of target
the same? Measure the median radial distance as a function of *both* presynaptic interneuron subclass
and postsynaptic target class. For PTC and DTC cells, how does their medial targeting distance vary with
target?


In [ ]:
# Your code here

---
### Optional — the denominator problem

*Not needed for anything above, but useful for thinking about the data.*

Every distance histogram in this module is partly a picture of **where the dendrite is**. More
synapses at 100 µm may just mean more dendrite at 100 µm. It did not distort the comparisons
here, because two subclasses measured on the same cells share the same denominator and it
largely cancels — but the moment you compare *across* cells, or ask an absolute question like
"how dense is inhibition on the tuft", it stops cancelling.

The fix on a cell with a skeleton is to divide by **µm of dendrite per bin**. Run the cell
below and compare the middle panel to the right one: the raw counts are dominated by the shape
of the denominator.

In [ ]:
# Your code here

**Where this goes next.** Two threads worth a project:

1. The denominator here is *this cell's* dendrite. To compare across cells you need a
   denominator that travels — excitatory synapse density is one candidate, since spiny
   dendrite carries them at a roughly steady rate, and it needs no skeleton at all.
2. Everything above treats "distance from soma" as the axis. Electrotonically what matters is
   path length *and* diameter. `cell.skeleton` carries radius; a cable-weighted distance would
   be a genuinely better axis and nobody has run it on this dataset.

### Optional — is the signature a property of the cell, the subclass, or the pair?

*Harder than it looks, and the reason is a lesson about summary statistics.*


In [ ]:
# Your code here

**Read the ratio, then read the table underneath it, and notice they disagree.**

Taken at face value the ratio is unimpressive — barely above 1, which would say subclass
predicts almost nothing. But scan the per-cell table: most PTCs sit tightly in the teens and
twenties of microns, the DTCs between roughly 60 and 90. Those groups barely overlap.

**Two cells are doing the damage, and they are the two worst-sampled ones** — a PTC out at
DTC-like distances and a DTC in at PTC-like distances, both estimated from **eighteen
targets** against one to four hundred for everyone else. Because `std` squares deviations,
two bad estimates sink the ratio by themselves. Requiring thirty targets moves it from about
**1.2 to nearly 9.**

That is not p-hacking; it is declining to average in estimates you already know are
unreliable. Decide the threshold before you look, and say you applied one.

**The habit worth taking away is to plot before you summarise.** Figure 9 showed the
separation immediately; no variance ratio was needed to see it. Then ask the biological
question: is that odd cell mis-typed, badly reconstructed, or a real variant?

## Summary

1. **Excitatory input arrives on spines across the whole dendrite; inhibitory input arrives on
   shaft and soma** — and the soma is essentially inhibitory-only real estate.
2. **Which inhibitory subclass a synapse comes from predicts where it lands.** PTC within a
   few tens of microns of the soma, DTC several times further out and leaning apical.
3. **That is a property of the presynaptic cell.** One PTC does the same thing to hundreds of
   different targets.

And three habits worth more than the result: **filter for the failure mode your question
actually has**; **give every count a denominator** (Figure 11); **calibrate your shortcuts**
(Figure 7 is why Figure 8 is allowed to exist).

### What the names meant

Now they can be unpacked. `PTC` is **perisomatic targeting**, `DTC` **dendrite targeting**,
`ITC` **inhibitory targeting**, `STC` **sparse targeting** — the nomenclature of
Schneider-Mizell et al. The classifier that assigned them never saw a synapse; it worked from
soma, nucleus, dendrite and spine morphology alone. So what you measured is that morphology
predicts targeting well enough that the names are earned.

Two things that buys you. First, it is a real prediction rather than a definition — the labels
could have failed to separate, and you would have seen that. Second, it puts a number on
Module 1's caveat that these labels "might align more with molecular class than exact
targeting for a given neuron, particularly among ITC/VIP cells." Your `ITC` count — nine
synapses on the exemplar — is exactly where that caveat bites.

### Exercises

1. Chandelier cells target the axon initial segment. The compartment table has no AIS class —
   so "what this dataset cannot tell you" is a legitimate finding. What would you need? What would the r, theta plot
   look like for a chandelier cell?
3. Does synapse `size` vary with compartment, or with presynaptic subclass?
4. Normalize r by target soma radius (`cell_df.volume`) and redo Figure 10. Does the
   L5ET/L4IT difference survive?
5. Pick your own interneuron, unscreened, and run it through `outputs_in_target_frame`. Does
   anything look wrong? How would you know?